# Team 08 - Python Ninjas: Cardiac Failure Dataset Cleaning & Preprocessing Pipeline

## Python Hackathon September 2026 Submission
- **Team Number:** 08
- **Team Name:** Python Ninjas
- **Category:** Category 1 - Data Preprocessing & Cleaning
- **Benchmark Dataset:** [PhysioNet Heart Failure Zigong Dataset (v1.3)](https://www.physionet.org/content/heart-failure-zigong/1.3/)
- **Repository Reference:** [NumpyNinja2025/Python_Hackathon](https://github.com/NumpyNinja2025/Python_Hackathon)

---

### Column Dropping Policy:
- **ONLY Zero-Impact / Zero-Variance Columns are dropped:**
  1. `labs.csv` -> `cholinesterase` (100% missing / 0 unique values)
  2. `labs.csv` -> `body_temperature_blood_gas` (100% single constant value 37.0)
  3. `patienthistory.csv` -> `leukemia` (100% single constant value 0)
- **All remaining columns are RETAINED** (including rare comorbidity indicators, diagnostic panels, ultrasound measurements, and respiratory support).

---

### 13-Step Preprocessing & Cleaning Architecture
1. **Step 1:** Load and Inspect Data & Data Dictionary
2. **Step 2:** Missing Value Analysis & Semantic Classification
3. **Step 3:** Duplicate Row and Join Key Audit
4. **Step 4:** Cardinality & Zero-Variance Analysis
5. **Step 5:** Column Drop Decisions (Zero-Impact Columns Only)
6. **Step 6:** Invalid / Physiologically Impossible Value Detection & Rounding
7. **Step 7:** Continuous Feature Outlier Detection (IQR & Z-Score)
8. **Step 8:** Clinically-Guided Missing Value Imputation
9. **Step 9:** Data Type Conversion & String Standardization
10. **Step 10:** Data Normalization Demonstration (Min-Max Scaling & Z-Score Standardization)
11. **Step 11:** Transactional Feature Aggregation & Relational Merging
12. **Step 12:** Predictive Modeling Target Leakage Audit
13. **Step 13:** Consolidated Summary Report & Cleaned Dataset Export

In [1]:
# ==============================================================================
# ENVIRONMENT SETUP & CONFIGURATION
# ==============================================================================
import os
import sys
import warnings
import numpy as np
import pandas as pd
from scipy import stats

# Suppress non-critical runtime warnings
warnings.filterwarnings('ignore')

# Configure Pandas display options
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

# Helper function for safe file saving
def save_csv_safe(df, path):
    try:
        df.to_csv(path, index=False)
        print(f'  Saved: {path}')
    except PermissionError:
        alt_path = path.replace('.csv', '_updated.csv')
        df.to_csv(alt_path, index=False)
        print(f'  [LOCKED by Excel] Saved to: {alt_path}')

# Configure dataset paths
BASE_DIR = r'Python_Hackathon_Sep_2026'
DATA_DIR = os.path.join(BASE_DIR, 'cardiac_failure')
DICT_PATH = os.path.join(BASE_DIR, 'Cardiac_failure_data_dictionary.xlsx')

print('Environment initialized successfully.')
print(f'Data Directory: {DATA_DIR}')

Environment initialized successfully.
Data Directory: Python_Hackathon_Sep_2026\cardiac_failure


## STEP 1: LOAD AND INSPECT
- Load all 7 clinical CSV files into Pandas DataFrames
- Profile row counts, column counts, data types, and primary key distributions
- Identify common join key (`inpatient_number`)
- Distinguish between patient-level (1:1) and transactional (1:N) tables
- Inspect the data dictionary for clinical metadata definitions

In [2]:
# 1.1 Load all raw CSV files
raw_files = [f for f in sorted(os.listdir(DATA_DIR)) if f.endswith('.csv') and not f.startswith(('cardiac_failure_', '8_Python_'))]
raw_dfs = {f: pd.read_csv(os.path.join(DATA_DIR, f)) for f in raw_files}

# 1.2 Inspect shapes and entity types
print('=' * 95)
print('STEP 1: DATASET PROFILING & ENTITY LEVEL IDENTIFICATION')
print('=' * 95)
for f, df in raw_dfs.items():
    n_unique = df['inpatient_number'].nunique()
    entity_type = 'Transactional (1:N)' if f == 'patient_precriptions.csv' else 'Patient-Level (1:1)'
    print(f'File: {f:<30} | Rows: {df.shape[0]:<6} | Cols: {df.shape[1]:<4} | Unique Inpatients: {n_unique:<5} | Type: {entity_type}')

# 1.3 Check Data Dictionary
if os.path.exists(DICT_PATH):
    xl = pd.ExcelFile(DICT_PATH)
    print(f'\nData Dictionary Found: {os.path.basename(DICT_PATH)}')
    print(f'Available Sheets: {xl.sheet_names}')
    for s in xl.sheet_names:
        df_sheet = xl.parse(s)
        print(f'  - Sheet "{s}": {df_sheet.shape[0]} variable definitions')

STEP 1: DATASET PROFILING & ENTITY LEVEL IDENTIFICATION
File: cardiac_complications.csv      | Rows: 2008   | Cols: 14   | Unique Inpatients: 2008  | Type: Patient-Level (1:1)
File: demography.csv                 | Rows: 2009   | Cols: 7    | Unique Inpatients: 2009  | Type: Patient-Level (1:1)
File: hospitalization_discharge.csv  | Rows: 2008   | Cols: 21   | Unique Inpatients: 2008  | Type: Patient-Level (1:1)
File: labs.csv                       | Rows: 2008   | Cols: 107  | Unique Inpatients: 2008  | Type: Patient-Level (1:1)
File: patient_precriptions.csv       | Rows: 15362  | Cols: 2    | Unique Inpatients: 2007  | Type: Transactional (1:N)
File: patienthistory.csv             | Rows: 2008   | Cols: 17   | Unique Inpatients: 2008  | Type: Patient-Level (1:1)
File: responsivenes.csv              | Rows: 2008   | Cols: 6    | Unique Inpatients: 2008  | Type: Patient-Level (1:1)

Data Dictionary Found: Cardiac_failure_data_dictionary.xlsx
Available Sheets: ['Data Definition']
  - S

## STEP 2: MISSING VALUE ANALYSIS
- Calculate missing count and percentage for every column across all datasets
- Classify missingness semantics:
  - **(a) True Missing:** Unintended data entry gaps
  - **(b) Structurally Missing:** Value is absent because an event did not occur (e.g. `time_of_death` is null because the patient survived)
  - **(c) Test-Not-Ordered:** Specialized lab or echocardiographic panels not ordered for this clinical admission
- Flag columns with 50–80% missing and >80% missing

In [3]:
dfs = {f: df.copy() for f, df in raw_dfs.items()}

missing_records = []
flagged_50_80 = []
flagged_over_80 = []

for f, df in dfs.items():
    for col in df.columns:
        if col == 'inpatient_number':
            continue
        n_miss = int(df[col].isnull().sum())
        pct_miss = float((n_miss / len(df)) * 100)
        
        # Clinical semantic classification
        if col in ['time_of_death__days_from_admission', 'readmission_time_days_from_admission', 
                   'time_to_emergency_department_within_6_months', 'respiratory_support']:
            miss_type = '(b) Structurally Missing (Event-driven / Absent Event)'
        elif 'gas' in col or col in ['ph', 'lactate', 'anion_gap', 'homocysteine', 'apolipoprotein_a', 
                                     'apolipoprotein_b', 'lipoprotein', 'erythrocyte_sedimentation_rate',
                                     'high_sensitivity_protein', 'lvef', 'mitral_valve_ems', 'mitral_valve_ams',
                                     'ea', 'tricuspid_valve_return_velocity', 'tricuspid_valve_return_pressure',
                                     'glutamic_oxaliplatin', 'inorganic_phosphorus', 'serum_magnesium',
                                     'nucleotidase', 'fucosidase', 'total_bile_acid']:
            miss_type = '(c) Test-Not-Ordered (Diagnostic Panel)'
        else:
            miss_type = '(a) True Missing (Data Entry Gap)'
            
        if pct_miss > 80.0:
            flagged_over_80.append((f, col, pct_miss, miss_type))
        elif pct_miss >= 50.0:
            flagged_50_80.append((f, col, pct_miss, miss_type))
            
        if n_miss > 0:
            missing_records.append({
                'File': f, 'Column': col, 'Missing_Count': n_miss, 
                'Missing_Pct': round(pct_miss, 2), 'Classification': miss_type
            })

df_missing_summary = pd.DataFrame(missing_records).sort_values(by='Missing_Pct', ascending=False)
print('Top 15 Columns with Highest Missingness:')
print(df_missing_summary.head(15).to_string(index=False))
print(f'\nTotal Columns >80% Missing: {len(flagged_over_80)}')
print(f'Total Columns 50-80% Missing: {len(flagged_50_80)}')

Top 15 Columns with Highest Missingness:
                         File                             Column  Missing_Count  Missing_Pct                                         Classification
                     labs.csv                     cholinesterase           2008       100.00                      (a) True Missing (Data Entry Gap)
hospitalization_discharge.csv                respiratory_support           1966        97.91 (b) Structurally Missing (Event-driven / Absent Event)
hospitalization_discharge.csv time_of_death__days_from_admission           1964        97.81 (b) Structurally Missing (Event-driven / Absent Event)
                     labs.csv                       homocysteine           1862        92.73                (c) Test-Not-Ordered (Diagnostic Panel)
                     labs.csv                   apolipoprotein_b           1832        91.24                (c) Test-Not-Ordered (Diagnostic Panel)
                     labs.csv                   apolipoprotein_a       

## STEP 3: DUPLICATE CHECK
- Audit full-row duplicate records across all datasets
- Audit join keys (`inpatient_number`)
- Check transactional (patient_id, drug_name) prescription pairs
- Remove full duplicate rows only

In [4]:
print('=' * 95)
print('STEP 3: DUPLICATE ROW AND JOIN KEY AUDIT')
print('=' * 95)
duplicates_removed = {}
for f, df in dfs.items():
    full_dups = int(df.duplicated().sum())
    key_dups = int(df.duplicated(subset=['inpatient_number']).sum())
    if full_dups > 0:
        dfs[f] = df.drop_duplicates()
    duplicates_removed[f] = full_dups
    print(f'File: {f:<30} | Full Duplicates: {full_dups:<4} | Duplicate Patient Keys: {key_dups:<5}')

rx_dups = dfs['patient_precriptions.csv'].duplicated(subset=['inpatient_number', 'drug_name']).sum()
print(f'Prescriptions (inpatient_number, drug_name) duplicate pairs: {rx_dups}')

STEP 3: DUPLICATE ROW AND JOIN KEY AUDIT
File: cardiac_complications.csv      | Full Duplicates: 0    | Duplicate Patient Keys: 0    
File: demography.csv                 | Full Duplicates: 0    | Duplicate Patient Keys: 0    
File: hospitalization_discharge.csv  | Full Duplicates: 0    | Duplicate Patient Keys: 0    
File: labs.csv                       | Full Duplicates: 0    | Duplicate Patient Keys: 0    
File: patient_precriptions.csv       | Full Duplicates: 0    | Duplicate Patient Keys: 13355
File: patienthistory.csv             | Full Duplicates: 0    | Duplicate Patient Keys: 0    
File: responsivenes.csv              | Full Duplicates: 0    | Duplicate Patient Keys: 0    
Prescriptions (inpatient_number, drug_name) duplicate pairs: 0


## STEP 4: CARDINALITY ANALYSIS
- Count unique values per column
- Identify constant columns (<=1 unique value) -> Zero Information
- Identify near-constant columns (>99.5% dominant category)

In [5]:
constant_cols = []
near_constant_cols = []

for f, df in dfs.items():
    for col in df.columns:
        n_uniq = df[col].nunique(dropna=True)
        n_valid = df[col].count()
        
        if n_uniq <= 1:
            constant_cols.append((f, col, n_uniq, 'Constant Column (Zero Variance)'))
        elif n_valid > 0:
            top_pct = df[col].value_counts(normalize=True).iloc[0]
            if top_pct >= 0.995:
                near_constant_cols.append((f, col, f'{top_pct*100:.2f}%', df[col].value_counts().index[0]))

print('--- Constant Columns (Zero Information) ---')
for item in constant_cols:
    print(f'  - [{item[0]}] "{item[1]}" - Unique values: {item[2]}')

print('\n--- Near-Constant Columns (>99.5% Single Category) ---')
for item in near_constant_cols:
    print(f'  - [{item[0]}] "{item[1]}" (Dominant Value "{item[3]}": {item[2]})')

--- Constant Columns (Zero Information) ---
  - [labs.csv] "cholinesterase" - Unique values: 0
  - [labs.csv] "body_temperature_blood_gas" - Unique values: 1
  - [patienthistory.csv] "leukemia" - Unique values: 1

--- Near-Constant Columns (>99.5% Single Category) ---
  - [patienthistory.csv] "connective_tissue_disease" (Dominant Value "0": 99.80%)
  - [patienthistory.csv] "malignant_lymphoma" (Dominant Value "0": 99.95%)
  - [patienthistory.csv] "aids" (Dominant Value "0": 99.80%)
  - [patienthistory.csv] "acute_renal_failure" (Dominant Value "0": 99.65%)


## STEP 5: COLUMN DROP DECISIONS
**Dropping Policy:**
1. **Zero-Variance / Zero-Impact Columns (Rule 1):**
   - `cholinesterase` in `labs.csv`: 100% missing / 0 unique values
   - `body_temperature_blood_gas` in `labs.csv`: 100% constant 37.0
   - `leukemia` in `patienthistory.csv`: 100% constant 0
2. **Redundant Post-Discharge Time-to-Event Columns (Rule 2):**
   - `readmission_time_days_from_admission` in `hospitalization_discharge.csv`: Post-discharge duration timestamp causing target leakage
   - `time_to_emergency_department_within_6_months` in `hospitalization_discharge.csv`: 100% duplicate timestamp of readmission time ($r=1.00$); the binary endpoint `return_to_emergency_department_within_6_months` is retained

**All other clinical columns** (comorbidities, diagnostic panels, ultrasound, respiratory support) are **retained**.

In [6]:
drop_rules = {
    'labs.csv': {
        'cholinesterase': ('Rule 1 (Zero Variance / 100% Missing)', 'All records are NaN (0 unique values) - carries zero mathematical signal'),
        'body_temperature_blood_gas': ('Rule 1 (Zero Variance / Constant)', '100% single value 37.0 - zero variance (blood gas uncalibrated default)')
    },
    'patienthistory.csv': {
        'leukemia': ('Rule 1 (Zero Variance / Constant)', '100% value 0 across all 2,008 patients - zero variance')
    },
    'hospitalization_discharge.csv': {
        'readmission_time_days_from_admission': ('Rule 2 (Post-Discharge Target Leakage)', 'Post-discharge timestamp; binary readmission targets retained'),
        'time_to_emergency_department_within_6_months': ('Rule 2 (Redundant Duplicate & Target Leakage)', '100% duplicate of readmission time (r=1.00); binary endpoint return_to_emergency_department_within_6_months retained')
    }
}

dropped_log = []
for f, rules in drop_rules.items():
    if f in dfs:
        for col, (rule, reason) in rules.items():
            if col in dfs[f].columns:
                dfs[f] = dfs[f].drop(columns=[col])
                dropped_log.append({'File': f, 'Column': col, 'Rule': rule, 'Reason': reason})
                print(f'DROPPED: [{f}] "{col}" | {rule} | Reason: {reason}')

print(f'\nTotal Columns Dropped: {len(dropped_log)}')
print('All other columns (comorbidities, diagnostic panels, ultrasound, respiratory support) are RETAINED.')

DROPPED: [labs.csv] "cholinesterase" | Rule 1 (Zero Variance / 100% Missing) | Reason: All records are NaN (0 unique values) - carries zero mathematical signal
DROPPED: [labs.csv] "body_temperature_blood_gas" | Rule 1 (Zero Variance / Constant) | Reason: 100% single value 37.0 - zero variance (blood gas uncalibrated default)
DROPPED: [patienthistory.csv] "leukemia" | Rule 1 (Zero Variance / Constant) | Reason: 100% value 0 across all 2,008 patients - zero variance
DROPPED: [hospitalization_discharge.csv] "readmission_time_days_from_admission" | Rule 2 (Post-Discharge Target Leakage) | Reason: Post-discharge timestamp; binary readmission targets retained
DROPPED: [hospitalization_discharge.csv] "time_to_emergency_department_within_6_months" | Rule 2 (Redundant Duplicate & Target Leakage) | Reason: 100% duplicate of readmission time (r=1.00); binary endpoint return_to_emergency_department_within_6_months retained

Total Columns Dropped: 5
All other columns (comorbidities, diagnostic pane

## STEP 6: INVALID VALUE DETECTION & ROUNDING
- Identify and sanitize physiologically impossible values:
  - Weight <= 0 kg -> NaN
  - Height < 0.5 m -> NaN
  - BMI <= 0 or > 80 -> NaN (recomputed from valid weight/height)
  - Baseline Vitals: Pulse <= 0, Respiration <= 0, SBP <= 0, DBP <= 0 -> NaN
  - Clinical Chemistry: Anion Gap < 0 mmol/L -> NaN
- **Strictly Enforce Precision Rounding Requirements:**
  1. **BMI:** 2 decimal places
  2. **weight:** 1 decimal place
  3. **map_value:** 2 decimal places
  4. **creatine_kinase_isoenzyme_to_creatine_kinase:** 2 decimal places
  5. **hydroxybutyrate_dehydrogenase_to_lactate_dehydrogenase:** 1 decimal place

In [7]:
df_demo = dfs['demography.csv']
df_labs = dfs['labs.csv']

# 6.1 Demography Sanitization
inv_w = (df_demo['weight'] <= 0)
if inv_w.sum() > 0:
    print(f'Demography: weight <= 0 in {inv_w.sum()} rows -> Set to NaN')
    df_demo.loc[inv_w, 'weight'] = np.nan

inv_h = (df_demo['height'] < 0.5)
if inv_h.sum() > 0:
    print(f'Demography: height < 0.5m in {inv_h.sum()} rows -> Set to NaN')
    df_demo.loc[inv_h, 'height'] = np.nan

inv_bmi = (df_demo['bmi'] <= 0) | (df_demo['bmi'] > 80)
if inv_bmi.sum() > 0:
    print(f'Demography: BMI bounds error in {inv_bmi.sum()} rows -> Set to NaN')
    df_demo.loc[inv_bmi, 'bmi'] = np.nan

# Recompute BMI where valid weight and height exist
recomp = df_demo['bmi'].isnull() & df_demo['weight'].notnull() & df_demo['height'].notnull()
df_demo.loc[recomp, 'bmi'] = df_demo.loc[recomp, 'weight'] / (df_demo.loc[recomp, 'height'] ** 2)

# 6.2 Labs Sanitization (Vitals & Anion Gap)
for v in ['pulse', 'respiration', 'systolic_blood_pressure', 'diastolic_blood_pressure']:
    if v in df_labs.columns:
        inv_mask = (df_labs[v] <= 0)
        if inv_mask.sum() > 0:
            print(f'Labs: {v} <= 0 in {inv_mask.sum()} rows -> Set to NaN')
            df_labs.loc[inv_mask, v] = np.nan

if 'anion_gap' in df_labs.columns:
    inv_ag = (df_labs['anion_gap'] < 0)
    if inv_ag.sum() > 0:
        print(f'Labs: anion_gap < 0 in {inv_ag.sum()} rows -> Set to NaN')
        df_labs.loc[inv_ag, 'anion_gap'] = np.nan

# 6.3 Apply Specific Decimal Precision Rounding
df_demo['bmi'] = df_demo['bmi'].round(2)
df_demo['weight'] = df_demo['weight'].round(1)

if 'map_value' in df_labs.columns:
    df_labs['map_value'] = df_labs['map_value'].round(2)
if 'creatine_kinase_isoenzyme_to_creatine_kinase' in df_labs.columns:
    df_labs['creatine_kinase_isoenzyme_to_creatine_kinase'] = df_labs['creatine_kinase_isoenzyme_to_creatine_kinase'].round(2)
if 'hydroxybutyrate_dehydrogenase_to_lactate_dehydrogenase' in df_labs.columns:
    df_labs['hydroxybutyrate_dehydrogenase_to_lactate_dehydrogenase'] = df_labs['hydroxybutyrate_dehydrogenase_to_lactate_dehydrogenase'].round(1)

print('\nRequired Precision Rounding Enforced:')
print('  1. BMI -> 2 decimal places')
print('  2. weight -> 1 decimal place')
print('  3. map_value -> 2 decimal places')
print('  4. creatine_kinase_isoenzyme_to_creatine_kinase -> 2 decimal places')
print('  5. hydroxybutyrate_dehydrogenase_to_lactate_dehydrogenase -> 1 decimal place')

Demography: weight <= 0 in 3 rows -> Set to NaN
Demography: height < 0.5m in 3 rows -> Set to NaN
Demography: BMI bounds error in 7 rows -> Set to NaN
Labs: pulse <= 0 in 1 rows -> Set to NaN
Labs: respiration <= 0 in 1 rows -> Set to NaN
Labs: systolic_blood_pressure <= 0 in 3 rows -> Set to NaN
Labs: diastolic_blood_pressure <= 0 in 3 rows -> Set to NaN
Labs: anion_gap < 0 in 2 rows -> Set to NaN

Required Precision Rounding Enforced:
  1. BMI -> 2 decimal places
  2. weight -> 1 decimal place
  3. map_value -> 2 decimal places
  4. creatine_kinase_isoenzyme_to_creatine_kinase -> 2 decimal places
  5. hydroxybutyrate_dehydrogenase_to_lactate_dehydrogenase -> 1 decimal place


## STEP 7: OUTLIER DETECTION (CONTINUOUS NUMERIC ONLY)
- Exclude binary indicator columns (`nunique <= 2`), categorical codes, and IDs
- Calculate IQR bounds: $[Q1 - 1.5 \times IQR, Q3 + 1.5 \times IQR]$
- Calculate Z-Scores: $|z| > 3$
- Clinically valid extreme biomarker values (e.g. BNP, Troponin) reflect acute cardiac crisis and are retained

In [8]:
outlier_results = []
for f, df in dfs.items():
    if f == 'patient_precriptions.csv':
        continue
    for col in df.select_dtypes(include=[np.number]).columns:
        if col == 'inpatient_number' or df[col].nunique(dropna=True) <= 2:
            continue
            
        series = df[col].dropna()
        if len(series) < 30:
            continue
            
        q1, q3 = series.quantile(0.25), series.quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        iqr_outs = series[(series < lower) | (series > upper)]
        z_outs = series[np.abs(stats.zscore(series)) > 3.0]
        
        outlier_results.append({
            'File': f, 'Column': col, 'IQR_Lower': round(lower, 2), 'IQR_Upper': round(upper, 2),
            'IQR_Outliers': len(iqr_outs), 'IQR_Pct': round(len(iqr_outs)/len(series)*100, 2),
            'Z_Outliers': len(z_outs)
        })

df_outliers = pd.DataFrame(outlier_results).sort_values(by='IQR_Pct', ascending=False)
print('Top 15 Continuous Numeric Columns by Outlier Percentage:')
print(df_outliers.head(15).to_string(index=False))

Top 15 Continuous Numeric Columns by Outlier Percentage:
                         File                             Column  IQR_Lower  IQR_Upper  IQR_Outliers  IQR_Pct  Z_Outliers
                     labs.csv        platelet_distribution_width      15.10      17.50           266    13.96          63
                     labs.csv                               fio2      33.00      33.00           248    12.35          15
                     labs.csv           high_sensitivity_protein     -34.70      68.50           115    12.22          21
                     labs.csv                        respiration      16.50      20.50           245    12.21          31
                     labs.csv          high_sensitivity_troponin      -0.12       0.26           225    11.66          11
                     labs.csv                            d_dimer      -1.28       4.24           213    11.58          21
hospitalization_discharge.csv time_of_death__days_from_admission     -23.12      43.88   

## STEP 8: MISSING VALUE IMPUTATION
- **True Missing Numeric (Demographics, Routine Vitals & Labs):** Impute with Median (robust to outliers)
- **True Missing Categorical:** Impute with Mode
- **Comorbidities:** Missing records imputed as absent ($0$)
- **Test-Not-Ordered / Structurally Missing:** Retained as NaN (allowing tree models like XGBoost to leverage missingness signal)

In [9]:
# 8.1 Demographics Imputation
for num_col in ['weight', 'height', 'bmi']:
    df_demo[num_col] = df_demo[num_col].fillna(df_demo[num_col].median())

for cat_col in ['gender', 'occupation', 'agecat']:
    df_demo[cat_col] = df_demo[cat_col].fillna(df_demo[cat_col].mode()[0])

# 8.2 Comorbidity History Imputation (Retaining all rare comorbidity columns)
df_hist = dfs['patienthistory.csv']
for col in df_hist.columns:
    if col == 'inpatient_number':
        continue
    if col == 'cci_score':
        df_hist[col] = df_hist[col].fillna(df_hist[col].median())
    elif df_hist[col].dtype == 'object' or isinstance(df_hist[col].dtype, pd.StringDtype):
        df_hist[col] = df_hist[col].fillna(df_hist[col].mode()[0])
    else:
        df_hist[col] = df_hist[col].fillna(0)

# 8.3 Hospitalization Emergency Return Flag
df_hosp = dfs['hospitalization_discharge.csv']
if 'return_to_emergency_department_within_6_months' in df_hosp.columns:
    df_hosp['return_to_emergency_department_within_6_months'] = df_hosp['return_to_emergency_department_within_6_months'].fillna(df_hosp['return_to_emergency_department_within_6_months'].mode()[0])

# 8.4 Routine Lab Tests (<10% missing)
routine_labs = [c for c in df_labs.select_dtypes(include=[np.number]).columns if c != 'inpatient_number' and df_labs[c].isnull().mean() < 0.10]
for col in routine_labs:
    df_labs[col] = df_labs[col].fillna(df_labs[col].median())

print('Imputation strategy executed across all datasets.')

Imputation strategy executed across all datasets.


## STEP 9: DATA TYPE CONVERSION & STRING STANDARDIZATION
- Convert `admission_date` strings to proper datetime objects
- Standardize text casing and strip whitespace across categorical columns
- Cast comorbidity binary flags to integer

In [10]:
# 9.1 Datetime conversion
df_hosp['admission_date'] = pd.to_datetime(df_hosp['admission_date'], errors='coerce')

# 9.2 String standardization
for f, df in dfs.items():
    for col in df.select_dtypes(include=['object']).columns:
        if col != 'admission_date':
            df[col] = df[col].astype(str).str.strip().str.title()

# 9.3 Cast comorbidity flags to integer
for col in ['peptic_ulcer_disease', 'moderate_to_severe_chronic_kidney_disease', 'liver_disease']:
    if col in df_hist.columns:
        df_hist[col] = df_hist[col].astype(int)

print('Data type conversions and string standardization completed.')

Data type conversions and string standardization completed.


## STEP 10: DATA NORMALIZATION (DEMONSTRATION & EDA)
Demonstrate continuous numeric feature scaling:
- **(a) Min-Max Normalization:** Scaled to $[0, 1]$
- **(b) Z-Score Standardization:** $\mu = 0, \sigma = 1$

> **Machine Learning Note:** For downstream predictive modeling, scalers should be fit exclusively on the training split to prevent data leakage.

In [11]:
norm_candidates = []
for f, df in dfs.items():
    if f == 'patient_precriptions.csv':
        continue
    for col in df.select_dtypes(include=[np.number]).columns:
        if col != 'inpatient_number' and df[col].nunique(dropna=True) > 15:
            norm_candidates.append((f, col))

norm_stats = []
for f, col in norm_candidates:
    s = dfs[f][col].dropna()
    s_min, s_max, s_mean, s_std = s.min(), s.max(), s.mean(), s.std()
    if s_max > s_min:
        s_minmax = (s - s_min) / (s_max - s_min)
        s_zscore = (s - s_mean) / s_std
        norm_stats.append({
            'File': f, 'Column': col,
            'Orig_Min': round(s_min, 2), 'Orig_Max': round(s_max, 2),
            'Orig_Mean': round(s_mean, 2), 'Orig_Std': round(s_std, 2),
            'MinMax_Min': round(s_minmax.min(), 2), 'MinMax_Max': round(s_minmax.max(), 2),
            'Z_Mean': round(s_zscore.mean(), 2), 'Z_Std': round(s_zscore.std(), 2)
        })

df_norm_summary = pd.DataFrame(norm_stats)
print(f'Total continuous numeric columns normalized: {len(df_norm_summary)}')
print('Sample of normalized features (Before & After Stats):')
print(df_norm_summary.head(10).to_string(index=False))

Total continuous numeric columns normalized: 114
Sample of normalized features (Before & After Stats):
                     File                                     Column  Orig_Min  Orig_Max  Orig_Mean  Orig_Std  MinMax_Min  MinMax_Max  Z_Mean  Z_Std
cardiac_complications.csv                                       lvef      5.00     82.00      50.68     13.22         0.0         1.0    -0.0    1.0
cardiac_complications.csv left_ventricular_end_diastolic_diameter_lv      0.30     88.00      53.11     10.92         0.0         1.0    -0.0    1.0
cardiac_complications.csv                           mitral_valve_ems      0.03    409.00       4.85     38.59         0.0         1.0    -0.0    1.0
cardiac_complications.csv                           mitral_valve_ams      0.06    408.00       4.09     34.60         0.0         1.0    -0.0    1.0
cardiac_complications.csv                                         ea      0.06     21.30       1.29      1.29         0.0         1.0    -0.0    1.0
car

## STEP 11: RELATIONAL MERGING & TRANSACTIONAL AGGREGATION
- **Transactional Table (`patient_precriptions.csv`):** Aggregate to patient-level features (total prescription count, unique drug classes count, one-hot encoded major cardiovascular drug classes)
- **Orphan Record Audit:** Detect and remove unlinked orphan records (e.g. `inpatient_number = 5`)
- **Relational Merge:** Outer join patient-level tables on `inpatient_number` into a unified Master analytical DataFrame
- **Re-enforce Decimal Precisions:** Explicitly ensure all 5 specified precision roundings are intact

In [12]:
# 11.1 Aggregate transactional prescriptions table
df_rx = dfs['patient_precriptions.csv']
rx_counts = df_rx.groupby('inpatient_number')['drug_name'].count().rename('total_prescriptions_count')
rx_unique = df_rx.groupby('inpatient_number')['drug_name'].nunique().rename('unique_drugs_count')
rx_pivoted = df_rx.pivot_table(index='inpatient_number', columns='drug_name', aggfunc=lambda x: 1, fill_value=0)
rx_pivoted.columns = ['rx_' + c.lower().replace(' ', '_')[:35] for c in rx_pivoted.columns]
df_rx_agg = pd.concat([rx_counts, rx_unique, rx_pivoted], axis=1).reset_index()

# 11.2 Orphan Record Removal
orphan_ids = set(df_demo['inpatient_number']) - set(df_hosp['inpatient_number'])
if len(orphan_ids) > 0:
    print(f'Removing orphan record(s) with no clinical history: {orphan_ids}')
    df_demo = df_demo[~df_demo['inpatient_number'].isin(orphan_ids)]

# 11.3 Relational Master Merge (Retains all non-zero-impact features)
master_df = df_demo.merge(df_hosp, on='inpatient_number', how='inner')
master_df = master_df.merge(dfs['cardiac_complications.csv'], on='inpatient_number', how='left')
master_df = master_df.merge(df_hist, on='inpatient_number', how='left')
master_df = master_df.merge(dfs['responsivenes.csv'], on='inpatient_number', how='left')
master_df = master_df.merge(df_labs, on='inpatient_number', how='left')
master_df = master_df.merge(df_rx_agg, on='inpatient_number', how='left')

# Impute prescription features for patients with 0 prescriptions
master_df['total_prescriptions_count'] = master_df['total_prescriptions_count'].fillna(0).astype(int)
master_df['unique_drugs_count'] = master_df['unique_drugs_count'].fillna(0).astype(int)
rx_cols = [c for c in master_df.columns if c.startswith('rx_')]
master_df[rx_cols] = master_df[rx_cols].fillna(0).astype(int)

# 11.4 Strictly Enforce Specified Decimal Precisions on Master Dataset
master_df['bmi'] = master_df['bmi'].round(2)
master_df['weight'] = master_df['weight'].round(1)
if 'map_value' in master_df.columns:
    master_df['map_value'] = master_df['map_value'].round(2)
if 'creatine_kinase_isoenzyme_to_creatine_kinase' in master_df.columns:
    master_df['creatine_kinase_isoenzyme_to_creatine_kinase'] = master_df['creatine_kinase_isoenzyme_to_creatine_kinase'].round(2)
if 'hydroxybutyrate_dehydrogenase_to_lactate_dehydrogenase' in master_df.columns:
    master_df['hydroxybutyrate_dehydrogenase_to_lactate_dehydrogenase'] = master_df['hydroxybutyrate_dehydrogenase_to_lactate_dehydrogenase'].round(1)

print(f'\nMaster Merged Analytical Dataset Created: {master_df.shape[0]} rows x {master_df.shape[1]} columns')

Removing orphan record(s) with no clinical history: {5}

Master Merged Analytical Dataset Created: 2008 rows x 189 columns


## STEP 12: TARGET LEAKAGE AUDIT
Audit potential feature leakage before predictive modeling:
- **In-Hospital Outcomes:** `outcome_during_hospitalization`, `destinationdischarge`, `dischargeday`
- **Post-Discharge Outcome Targets:** `death_within_28_days`, `death_within_3_months`, `death_within_6_months`, `re_admission_within_28_days`, `3_months`, `6_months`
- **Time-to-Event Post-Discharge Timestamps:** `readmission_time_days_from_admission`, `time_to_emergency_department_within_6_months`

In [13]:
leakage_matrix = [
    {'Column': 'outcome_during_hospitalization', 'Category': 'In-Hospital Mortality', 'Risk': 'High', 'Guidance': 'Exclude when predicting at admission'},
    {'Column': 'destinationdischarge', 'Category': 'Discharge Disposition', 'Risk': 'High', 'Guidance': 'Exclude (reveals in-hospital outcome)'},
    {'Column': 'dischargeday', 'Category': 'Length of Stay', 'Risk': 'High', 'Guidance': 'Exclude for admission-time risk scoring'},
    {'Column': 'death_within_28_days / 3_months / 6_months', 'Category': 'Mortality Targets', 'Risk': 'High', 'Guidance': 'Use ONLY as prediction targets'},
    {'Column': 're_admission_within_28_days / 3_months / 6_months', 'Category': 'Readmission Targets', 'Risk': 'High', 'Guidance': 'Use ONLY as prediction targets'},
    {'Column': 'readmission_time_days / time_to_ed_6_months', 'Category': 'Time-to-Event Timestamps', 'Risk': 'Critical', 'Guidance': 'Exclude (presence implies event occurred)'}
]
print(pd.DataFrame(leakage_matrix).to_string(index=False))

                                           Column                 Category     Risk                                  Guidance
                   outcome_during_hospitalization    In-Hospital Mortality     High      Exclude when predicting at admission
                             destinationdischarge    Discharge Disposition     High     Exclude (reveals in-hospital outcome)
                                     dischargeday           Length of Stay     High   Exclude for admission-time risk scoring
       death_within_28_days / 3_months / 6_months        Mortality Targets     High            Use ONLY as prediction targets
re_admission_within_28_days / 3_months / 6_months      Readmission Targets     High            Use ONLY as prediction targets
      readmission_time_days / time_to_ed_6_months Time-to-Event Timestamps Critical Exclude (presence implies event occurred)


## STEP 13: SUMMARY REPORT & CLEANED DATASET EXPORT
- **Export the Single Cleaned Master CSV Dataset:** `8_Python_Ninjas_cleaned_data.csv`
- **Display the Consolidated Execution Summary Matrix**

In [14]:
# 13.1 Save Single Cleaned Master Dataset
official_cleaned_path = '8_Python_Ninjas_cleaned_data.csv'
save_csv_safe(master_df, official_cleaned_path)
save_csv_safe(master_df, os.path.join(DATA_DIR, '8_Python_Ninjas_cleaned_data.csv'))

print('=' * 95)
print('OFFICIAL HACKATHON DATASET EXPORT CONFIRMATION')
print('=' * 95)
print(f'Cleaned Master Dataset: {official_cleaned_path} (Shape: {master_df.shape})')

# 13.2 Consolidated Execution Summary Matrix
summary_data = [
    {'File': 'demography.csv', 'Original_Shape': '(2009, 7)', 'Cleaned_Shape': '(2008, 7)', 'Columns_Dropped': '0', 'Duplicates_Removed': '0', 'Invalid_Fixed': 'weight<=0 (3), height<0.5m (3), bmi bounds (7)', 'Outliers_Detected': 'Weight (11), BMI (27)'},
    {'File': 'hospitalization_discharge.csv', 'Original_Shape': '(2008, 21)', 'Cleaned_Shape': '(2008, 19)', 'Columns_Dropped': '2 (readmission_time_days_from_admission, time_to_emergency_department_within_6_months)', 'Duplicates_Removed': '0', 'Invalid_Fixed': 'admission_date parsed to datetime', 'Outliers_Detected': 'dischargeday (154)'},
    {'File': 'cardiac_complications.csv', 'Original_Shape': '(2008, 14)', 'Cleaned_Shape': '(2008, 14)', 'Columns_Dropped': '0 (ea, tricuspid_pressure retained)', 'Duplicates_Removed': '0', 'Invalid_Fixed': 'None', 'Outliers_Detected': 'lvef (1), lv_diameter (39)'},
    {'File': 'labs.csv', 'Original_Shape': '(2008, 107)', 'Cleaned_Shape': '(2008, 105)', 'Columns_Dropped': '2 (cholinesterase, body_temperature_blood_gas)', 'Duplicates_Removed': '0', 'Invalid_Fixed': 'pulse=0, resp=0, sbp=0, dbp=0, anion_gap<0', 'Outliers_Detected': 'Retained acute cardiac biomarkers'},
    {'File': 'patienthistory.csv', 'Original_Shape': '(2008, 17)', 'Cleaned_Shape': '(2008, 16)', 'Columns_Dropped': '1 (leukemia: 100% 0)', 'Duplicates_Removed': '0', 'Invalid_Fixed': 'Imputed nulls to 0 & cast to int', 'Outliers_Detected': 'N/A (Binary comorbidity flags)'},
    {'File': 'responsivenes.csv', 'Original_Shape': '(2008, 6)', 'Cleaned_Shape': '(2008, 6)', 'Columns_Dropped': '0', 'Duplicates_Removed': '0', 'Invalid_Fixed': 'None', 'Outliers_Detected': 'GCS low extremes (57)'},
    {'File': 'patient_precriptions.csv', 'Original_Shape': '(15362, 2)', 'Cleaned_Shape': 'Aggregated to (2007, 28)', 'Columns_Dropped': 'Aggregated transactional to patient-level features', 'Duplicates_Removed': '0', 'Invalid_Fixed': 'None', 'Outliers_Detected': 'Prescription counts'}
]

df_summary_report = pd.DataFrame(summary_data)
print('\n--- CONSOLIDATED PIPELINE EXECUTION SUMMARY MATRIX ---')
print(df_summary_report.to_string(index=False))
print('=' * 95)
print('DATA CLEANING & MASTER MERGE COMPLETED SUCCESSFULLY!')

  Saved: 8_Python_Ninjas_cleaned_data.csv
  Saved: Python_Hackathon_Sep_2026\cardiac_failure\8_Python_Ninjas_cleaned_data.csv
OFFICIAL HACKATHON DATASET EXPORT CONFIRMATION
Cleaned Master Dataset: 8_Python_Ninjas_cleaned_data.csv (Shape: (2008, 189))

--- CONSOLIDATED PIPELINE EXECUTION SUMMARY MATRIX ---
                         File Original_Shape            Cleaned_Shape                                                                        Columns_Dropped Duplicates_Removed                                  Invalid_Fixed                 Outliers_Detected
               demography.csv      (2009, 7)                (2008, 7)                                                                                      0                  0 weight<=0 (3), height<0.5m (3), bmi bounds (7)             Weight (11), BMI (27)
hospitalization_discharge.csv     (2008, 21)               (2008, 19) 2 (readmission_time_days_from_admission, time_to_emergency_department_within_6_months)                  0    